In [1]:
%load_ext autoreload
%autoreload 2


In [8]:
from pipeline import full_analysis
mol, res, texte = full_analysis("fulvene")
print(texte)


Énergies π (unités β) :
  OM 1 : ε = +2.115 β   ● 2e-
  OM 2 : ε = +1.000 β   ● 2e-
  OM 3 : ε = +0.618 β   ● 2e-
  OM 4 : ε = -0.254 β   ○
  OM 5 : ε = -1.618 β   ○
  OM 6 : ε = -1.861 β   ○

Charges π :
  C1 : q_π = -0.047
  C2 : q_π = -0.092
  C3 : q_π = -0.092
  C4 : q_π = -0.073
  C5 : q_π = -0.073
  C6 : q_π = +0.378

Ordres de liaison π :
  C1–C2 : p_π = 0.449
  C1–C3 : p_π = 0.449
  C1–C6 : p_π = 0.759
  C2–C4 : p_π = 0.778
  C3–C5 : p_π = 0.778
  C4–C5 : p_π = 0.520

Énergie π totale : -7.466 β


In [2]:


import molecule, huckel, settings, huckel_analysis, chch_pubchem
import numpy as np
from molecule import read_xyz

syms, xyz = read_xyz("pyridine.xyz")
#mol = molecule.Molecule(syms, xyz).guess_bonds().assign_types()
mol = molecule.Molecule(syms, xyz)

#for a in mol.atoms:
#    print(a.el, len(a.neighbors), a.atype, getattr(a, 'flag', ''))

#print(len(mol.bonds)) if hasattr(mol, 'bonds') else None
#print(settings.RCOV, getattr(settings, 'BOND_TOL', None))

#for i, a in enumerate(mol):
 #   print(f"{i+1:2d} {a.el:>2s}  type={a.atype!r}  npi={a.npi}  "
#          f"voisins={a.neighbors}  orders={a.orders}")
H,idx = huckel.build_hamiltonian(mol)   # ta fonction existante
#huckel.print_hamiltonian(mol, H)
huckel.print_hamiltonian(mol, H)
res = huckel.solve_huckel(mol)
print("\nÉnergies π :", np.round(res["e"],3),"n unités ß")


Hamiltonian
         N        C        C        C        C        C
------------------------------------------------------
 N |   -0.510   -1.020    0.000    0.000    0.000   -1.020
 C |   -1.020   -0.000   -1.000    0.000    0.000    0.000
 C |    0.000   -1.000   -0.000   -1.000    0.000    0.000
 C |    0.000    0.000   -1.000   -0.000   -1.000    0.000
 C |    0.000    0.000    0.000   -1.000   -0.000   -1.000
 C |   -1.020    0.000    0.000    0.000   -1.000   -0.000

Énergies π : [-2.128 -1.179 -1.     0.854  1.     1.943] n unités ß


In [6]:
# test_pubchem.py (ou une cellule de notebook)
import molecule, huckel, settings, huckel_analysis, chch_pubchem
import numpy as np
from molecule import read_xyz
mol_pubchem = chch_pubchem.mol_from_name("fulvene")
xyz_str = chch_pubchem.mol_to_xyz(mol_pubchem, "mol_pubchem")
syms, xyz = molecule.read_xyz("pyridine.xyz")

syms, xyz = molecule.read_xyz(text=xyz_str)     # ← text=, pas path=
mol = molecule.Molecule(syms, xyz)

#H,idx = huckel.build_hamiltonian(mol)   # ta fonction existante
#huckel.print_hamiltonian(mol, H)
res = huckel.solve_huckel(mol)
#print(res.keys())        # si c'est un dict
#print([n for n in dir(huckel_analysis) if not n.startswith('_')])

report, res_analysis = huckel_analysis.format_report(mol, res)   # ← res, pas res_analysis
print(report)
#print("\nÉnergies π :", np.round(res["e"],3),"n unités ß")



Énergies π (unités β) :
  OM 1 : ε = +2.115 β   ● 2e-
  OM 2 : ε = +1.000 β   ● 2e-
  OM 3 : ε = +0.618 β   ● 2e-
  OM 4 : ε = -0.254 β   ○
  OM 5 : ε = -1.618 β   ○
  OM 6 : ε = -1.861 β   ○

Charges π :
  C1 : q_π = -0.047
  C2 : q_π = -0.092
  C3 : q_π = -0.092
  C4 : q_π = -0.073
  C5 : q_π = -0.073
  C6 : q_π = +0.378

Ordres de liaison π :
  C1–C2 : p_π = 0.449
  C1–C3 : p_π = 0.449
  C1–C6 : p_π = 0.759
  C2–C4 : p_π = 0.778
  C3–C5 : p_π = 0.778
  C4–C5 : p_π = 0.520

Énergie π totale : -7.466 β


In [4]:
from huckel import huckel     # à la place de / en plus de  import huckel
import settings,importlib, molecule
with open("pyridine.xyz") as f:
    syms, xyz = read_xyz(f.read())
print(f"{len(syms)} atomes lus :", " ".join(syms))   # attendu : 11, N C×5 H×5

mol = Molecule(syms, xyz)
mol.guess_bonds()
mol.assign_types()

# --- 1. connectivité ---
print("\nLiaisons détectées :", sum(len(a.neighbors) for a in mol.atoms)//2,
      "(attendu : 11 = 6 cycle + 5 C–H)")
for i, a in enumerate(mol.atoms):
    for j in a.neighbors:
        if j > i:
            d = np.linalg.norm(xyz[i]-xyz[j])
            print(f"  {a.el}{i+1}–{mol.atoms[j].el}{j+1} : {d:.3f} Å")

# --- 2. typage ---
print("\n" + mol.summary())

# --- 3. Hückel ---
from huckel import huckel          # ← la FONCTION, pas le module
res = huckel(mol)
e, occ, c = res["e"], res["occ"], res["c"]

print(f"\nélectrons π = {mol.npz():.0f}        (attendu : 6)")
print(f"Eπ = {np.sum(occ*e):+.3f} |β|     (attendu ≈ +6.90)")
print(f"orbitales occupées = {np.sum(occ):.1f}  (attendu : 6.0 = 3 OM × 2 e⁻)")

# --- 4. charges π ---
P = c @ np.diag(occ) @ c.T
idx = mol.pi_idx
npi = np.array([mol.atoms[i].npi for i in idx])
q = npi - np.diag(P)
labels = ["N", "o", "m", "m'", "o", "p"]
for k, i in enumerate(idx):
    lab = labels[i] if len(idx) == 6 else f"{mol.atoms[i].el}{i+1}"
    print(f"  {lab:>3} ({mol.atoms[i].el}) : q = {q[k]:+.3f}")
print(f"Somme des charges : {q.sum():+.4f}   (attendu : 0)")

ImportError: cannot import name 'huckel' from 'huckel' (/scratch/nautilus/phileas/home/a02a2787@univ-amu.fr/HULIS/essai_HULIS/huckel.py)

In [ ]:
import numpy as np
from chch_pubchem import mol_from_name,mol_to_xyz
import molecule

if __name__ == "__main__":
    mol = mol_from_name("ethylene")
    xyz_str = mol_to_xyz(mol, "rrrrrr")
    syms, xyz = molecule.read_xyz(text=xyz_str)     # ← text=, pas path=
    m = molecule.Molecule(syms, xyz).guess_bonds().assign_types()


In [ ]:
print(syms,xyz)

In [ ]:
# 1. Vue d'ensemble
print(m.summary())

# 2. Atome par atome (ex. "C_ar", "N_am", etc.)
for atom in m.atoms:
    print(f"{atom.el}{atom.npi} : {atom.atype}")

# 3. En ligne, dans l'ordre du fichier
print([a.atype for a in m.atoms])